# 뇌 백질 고강도 병변 MRI 분할 (WMH 2017) 불균형 세그멘테이션 실험

---

## 1. 태스크 및 도메인
- **도메인**: 뇌 백질 고강도 병변 (White Matter Hyperintensity) MRI 세그멘테이션
- **모달리티**: MRI — FLAIR (primary) + T1 (auxiliary), 멀티채널
- **태스크**: Binary segmentation — 배경(0) / WMH 병변(1)
- **핵심 도전**: 이 프로젝트에서 가장 극단적인 불균형 도메인. 병변이 뇌 전체 대비 극소량.

## 2. 모델
- **아키텍처**: U-Net (ResNet34 백본)
- **사전학습**: ImageNet pretrained
- **선택 이유**: Binary segmentation 표준 베이스라인, 극심한 불균형에서 손실 함수 효과 검증
- **입력 구성**: 3채널 [FLAIR, T1, FLAIR] → ImageNet 인코더 3ch 채널 수 맞춤
- **출력**: 1채널 sigmoid → `to_2ch_logits()` 변환 후 손실 함수 적용

## 3. 데이터셋
- **이름**: WMH 2017 Challenge (White Matter Hyperintensity Segmentation)
- **규모**: 60 케이스, 3개 병원 — Utrecht / Singapore / Amsterdam
- **클래스 불균형**: BG:WMH = **~100:1 ~ 600:1** (극심한 불균형 — LiTS 종양 356:1 초과)
- **공식 분할**: 없음 → 케이스 단위 8:1:1 랜덤 분할 (random_state=42)
- **특이사항**: BG-only 슬라이스 30% 포함 (BG_ONLY_RATIO=0.3), 나머지는 WMH 존재 슬라이스

## 4. 데이터 준비 (협업자용)
> Cell 0 실행 시 자동으로 데이터가 다운로드됨. 별도 준비 불필요.

**취득 방법 (자동)**:
```python
kagglehub.dataset_download("farahmo/wmh-dataset")
```
공식 사이트 (현재 접속 불가): https://wmh.isi.uu.nl/

**폴더 구조** (자동 다운로드 후):
```
{kagglehub_cache}/
  {SiteName}/{SubjectID}/pre/
    FLAIR.nii.gz
    T1.nii.gz
  {SiteName}/{SubjectID}/
    wmh.nii.gz   ← 정답 마스크
```

## 5. 전처리 및 도메인 특이점
- Percentile 정규화 (1~99th percentile, foreground 복셀 기준) — MRI intensity 편차 보정
- 입력 3채널 [FLAIR, T1, FLAIR]: T1은 보조 채널, FLAIR이 주 채널
- 볼륨 단위 Train/Val/Test 분할 (슬라이스 단위 분할 시 data leakage 발생)
- BG-only 슬라이스 과다 포함 방지: WMH 슬라이스 수의 30%만 BG-only 슬라이스 허용

## 6. 실험 손실 함수 및 Optuna 탐색 범위
| 손실 함수 | 탐색 파라미터 | 탐색 범위 | Trials |
|-----------|-------------|----------|--------|
| `ce_dice` | — | — | — |
| `wce_dice` | — | — | — |
| `lwce_dice` | — | — | — |
| `plwce_dice` | alpha | **2.5 ~ 20.0** (극심한 불균형으로 상한 확장) | 30 |
| `pwce_dice` | alpha | 0.2 ~ 3.0 | 30 |
| `cb_dice` | — | — | — |
| `plwce_focal_dice` | alpha + gamma | alpha 2.5~20.0, gamma 0.5~5.0 | 60 |

## 7. SoTA 참고 (2026년 3월 기준)
| 방법 | Dice | Sensitivity | Specificity | 출처 |
|------|------|-------------|-------------|------|
| Robust-WMH-UNet (2026) | **0.768** | — | — | arXiv |
| Transformer-based (2025) | 0.720 | — | — | NeuroImage |
| nnU-Net 3D (2021) | ~0.800* | — | — | Nature Methods |
| 2D U-Net | ~0.750~0.790 | — | — | WMH Challenge |

> ⚠️ nnU-Net 0.800은 Utrecht 단일 사이트 기준; 3-site 평균은 낮을 수 있음.
> 본 연구 목표: U-Net baseline 대비 LWCE/PLWCE 계열 손실 함수의 개선 효과 검증.
> 평가 지표: Dice, Sensitivity, Specificity, AUC
> 결과 저장: `medical_data/results/WMH_Brain_Lesion_MRI/`

In [ ]:
# === Cell 0: 환경 설정 ===
import subprocess, sys

for pkg in ['segmentation-models-pytorch', 'optuna', 'nibabel', 'openpyxl', 'kagglehub']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import os, warnings, json, random, glob
warnings.filterwarnings('ignore')
os.environ['TQDM_DISABLE'] = '1'

import numpy as np
import nibabel as nib
import cv2
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import segmentation_models_pytorch as smp
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

sys.path.insert(0, '/root/imbalanced-data-LWCE/medical_data')
from custom_losses import get_loss_function
# --- 실험 설정 ---
DOMAIN      = 'wmh'
NUM_CLASSES = 2
CLASS_NAMES = ['Background', 'WMH']
IMG_SIZE    = 256
BATCH_SIZE  = 16
NUM_WORKERS = 4
SEED        = 42
BG_ONLY_RATIO = 0.3

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

RESULTS_DIR = '/root/imbalanced-data-LWCE/medical_data/results/WMH_Brain_Lesion_MRI'
os.makedirs(RESULTS_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('환경 설정 완료')

In [ ]:
# === Cell 1: 데이터 로드 ===
import kagglehub

# kagglehub으로 WMH 데이터셋 다운로드 (최초 1회 자동 캐시)
_kaggle_path = kagglehub.dataset_download("farahmo/wmh-dataset")
print("Path to dataset files:", _kaggle_path)

RAW_DIR = _kaggle_path

flair_check = glob.glob(os.path.join(RAW_DIR, '**', 'FLAIR.nii*'), recursive=True)
print(f'FLAIR 파일 {len(flair_check)}개 발견')

# --- 케이스 탐색: FLAIR + T1 + wmh 세트 ---
def find_wmh_cases(base_dir):
    """FLAIR, T1, wmh NIfTI 파일 세트 탐색. 다양한 WMH 데이터셋 구조 지원."""
    cases = []

    flair_files = glob.glob(os.path.join(base_dir, '**', 'FLAIR.nii*'), recursive=True)
    for flair_path in flair_files:
        case_dir  = os.path.dirname(flair_path)
        parent    = os.path.dirname(case_dir)

        t1_path   = os.path.join(case_dir, 'T1.nii.gz')
        if not os.path.exists(t1_path):
            t1_path = os.path.join(case_dir, 'T1.nii')

        wmh_candidates = [
            os.path.join(parent, 'wmh.nii.gz'),
            os.path.join(parent, 'wmh.nii'),
            os.path.join(case_dir, 'wmh.nii.gz'),
            os.path.join(case_dir, 'wmh.nii'),
            os.path.join(parent, 'lesion.nii.gz'),
            os.path.join(parent, 'mask.nii.gz'),
        ]
        wmh_path = next((p for p in wmh_candidates if os.path.exists(p)), None)

        if os.path.exists(t1_path) and wmh_path is not None:
            cases.append((flair_path, t1_path, wmh_path))

    return cases

cases = find_wmh_cases(RAW_DIR)
print(f'발견된 케이스 수 (FLAIR+T1+wmh): {len(cases)}')

if len(cases) == 0 and len(flair_check) > 0:
    print('케이스 매칭 실패. 데이터 구조를 확인하세요:')
    for root, dirs, files in os.walk(RAW_DIR):
        level = root.replace(RAW_DIR, '').count(os.sep)
        if level > 3: continue
        print('  ' * level + os.path.basename(root) + '/')
        if level <= 2:
            for f in files[:3]:
                print('  ' * (level + 1) + f)

# --- 슬라이스 전처리 (최초 1회) ---
SLICE_DIR = '/tmp/wmh_slices'
os.makedirs(SLICE_DIR, exist_ok=True)


def percentile_normalize(arr, p_low=1, p_high=99):
    """뇌 마스크 기준 percentile 정규화 → [0, 1]"""
    foreground = arr[arr > arr.mean() * 0.1]
    lo = np.percentile(foreground, p_low)  if len(foreground) > 0 else arr.min()
    hi = np.percentile(foreground, p_high) if len(foreground) > 0 else arr.max()
    arr = np.clip(arr, lo, hi)
    return ((arr - lo) / (hi - lo + 1e-8)).astype(np.float32)


existing = glob.glob(os.path.join(SLICE_DIR, '*.npz'))
if len(existing) < 100 and len(cases) > 0:
    print('슬라이스 전처리 중 (최초 1회)...')
    for case_idx, (flair_path, t1_path, wmh_path) in enumerate(tqdm(cases, desc='Processing')):
        flair_vol = nib.load(flair_path).get_fdata().astype(np.float32)
        t1_vol    = nib.load(t1_path).get_fdata().astype(np.float32)
        wmh_vol   = (nib.load(wmh_path).get_fdata() > 0.5).astype(np.int64)

        flair_vol = percentile_normalize(flair_vol)
        t1_vol    = percentile_normalize(t1_vol)

        n_slices = flair_vol.shape[2]
        for s in range(n_slices):
            flair_s = cv2.resize(flair_vol[:, :, s], (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
            t1_s    = cv2.resize(t1_vol[:, :, s],    (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
            wmh_s   = cv2.resize(wmh_vol[:, :, s].astype(np.uint8), (IMG_SIZE, IMG_SIZE),
                                  interpolation=cv2.INTER_NEAREST).astype(np.int64)

            # 3채널: [FLAIR, T1, FLAIR] — ImageNet 인코더 3ch 요구
            img_3ch = np.stack([flair_s, t1_s, flair_s], axis=0).astype(np.float32)
            fname   = os.path.join(SLICE_DIR, f'case{case_idx:03d}_s{s:04d}.npz')
            np.savez_compressed(fname, image=img_3ch, label=wmh_s)

    print('전처리 완료')
elif len(existing) >= 100:
    print(f'캐시 사용: {len(existing)}개 슬라이스 이미 존재')

# --- 슬라이스 로드 + WMH/BG-only 비율 조정 ---
all_slices = sorted(glob.glob(os.path.join(SLICE_DIR, '*.npz')))

wmh_slices = []
bg_slices  = []
for fp in all_slices:
    label = np.load(fp)['label']
    if label.sum() > 0:
        wmh_slices.append(fp)
    else:
        bg_slices.append(fp)

n_bg_include = min(len(bg_slices), int(len(wmh_slices) * BG_ONLY_RATIO))
bg_selected  = random.sample(bg_slices, n_bg_include)
slice_files  = sorted(wmh_slices + bg_selected)

print(f'WMH 있는 슬라이스: {len(wmh_slices)}')
print(f'BG-only 슬라이스 (선택): {n_bg_include} / {len(bg_slices)}')
print(f'총 학습 슬라이스: {len(slice_files)}')

# --- 볼륨 단위 Train/Val/Test 분할 (data leakage 방지, 8:1:1) ---
case_ids = sorted(set(int(os.path.basename(f).split('_')[0][4:]) for f in slice_files))
tr_case_ids, tmp_case_ids = train_test_split(case_ids, test_size=0.2, random_state=SEED)
val_case_ids, test_case_ids = train_test_split(tmp_case_ids, test_size=0.5, random_state=SEED)
tr_set   = set(tr_case_ids)
val_set  = set(val_case_ids)
test_set = set(test_case_ids)

tr_files   = [f for f in slice_files if int(os.path.basename(f).split('_')[0][4:]) in tr_set]
val_files  = [f for f in slice_files if int(os.path.basename(f).split('_')[0][4:]) in val_set]
test_files = [f for f in slice_files if int(os.path.basename(f).split('_')[0][4:]) in test_set]
print(f'Train: {len(tr_files)} slices ({len(tr_case_ids)} cases)')
print(f'Val  : {len(val_files)} slices ({len(val_case_ids)} cases)')
print(f'Test : {len(test_files)} slices ({len(test_case_ids)} cases)')

# --- Dataset 클래스 ---
class WMHDataset(Dataset):
    """
    WMH Brain Lesion MRI Dataset.
    Input:  (3, H, W) — [FLAIR, T1, FLAIR], percentile normalized [0, 1]
    Label:  (H, W)    — 0=Background, 1=WMH Lesion
    """
    def __init__(self, npz_files, augment=False):
        self.files   = npz_files
        self.augment = augment

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        data  = np.load(self.files[idx])
        image = data['image'].astype(np.float32)
        label = data['label'].astype(np.int64)

        if self.augment:
            if random.random() > 0.5:
                image = np.flip(image, axis=2).copy()
                label = np.fliplr(label).copy()
            if random.random() > 0.5:
                image = np.flip(image, axis=1).copy()
                label = np.flipud(label).copy()

        return torch.from_numpy(image), torch.from_numpy(label)


# --- DataLoader ---
train_loader = DataLoader(
    WMHDataset(tr_files,  augment=True),
    batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    WMHDataset(val_files, augment=False),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    WMHDataset(test_files, augment=False),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)
print('DataLoader 구성 완료')

In [ ]:
# === Cell 2: 모델 정의 (클래스 비율 계산 포함) ===
print('클래스 비율 계산 중 (학습 슬라이스 픽셀 단위)...')

class_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for fp in tqdm(tr_files, desc='Counting pixels'):
    label = np.load(fp)['label'].astype(np.int64)
    class_counts[0] += int((label == 0).sum())
    class_counts[1] += int((label == 1).sum())

class_counts = class_counts.tolist()
total = sum(class_counts)

print()
for c, (name, cnt) in enumerate(zip(CLASS_NAMES, class_counts)):
    print(f'  [{c}] {name:<12}: {cnt:>15,} pixels  ({100 * cnt / total:.4f}%)')

ratio = class_counts[0] / class_counts[1]
print(f'\nBG : WMH = {ratio:.1f} : 1  (극심한 불균형)')
print(f'\nclass_counts = {class_counts}')

In [ ]:
# === Cell 3: 학습 함수 (모델 + 유틸리티 포함) ===
#
# [SoTA 참고]
#   nnU-Net (3D, Isensee): Dice ≈ 0.800, Sensitivity ≈ 0.79, Specificity ≈ 0.99
#   2D U-Net (WMH 2017 top entries): Dice ≈ 0.75~0.79
#   출처: WMH 2017 Challenge Leaderboard (Kuijf et al., 2019, IEEE TMI)
#
# [선택 이유]
#   - 극심한 불균형(BG:FG>100:1) 환경에서 손실 함수 효과 분리 측정
#   - 3채널 [FLAIR, T1, FLAIR] → ImageNet pretrained ResNet34 활용 가능
#   - 경량 U-Net(ResNet34) 고정으로 손실 함수만 교체

def to_2ch_logits(p):
    """1채널 logit → 2채널 logit (rules.md §6-2)"""
    return torch.cat([-p, p], dim=1)


def build_model():
    """U-Net (ResNet34, ImageNet pretrained) — Binary WMH segmentation."""
    return smp.Unet(
        encoder_name    = 'resnet34',
        encoder_weights = 'imagenet',
        in_channels     = 3,
        classes         = 1,
        activation      = None,
    ).to(device)


def compute_val_dice(model, loader):
    """빠른 Val Dice — Optuna 및 학습 모니터링용"""
    model.eval()
    tp = fp = fn = 0
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            prob = torch.sigmoid(model(imgs)[:, 0])
            pred = (prob > 0.5).long()
            tp += ((pred == 1) & (masks == 1)).sum().item()
            fp += ((pred == 1) & (masks == 0)).sum().item()
            fn += ((pred == 0) & (masks == 1)).sum().item()
    return float(2 * tp / (2 * tp + fp + fn + 1e-8))


def compute_val_metrics(model, loader):
    """전체 Val 지표: Dice, Sensitivity, Specificity, AUC"""
    model.eval()
    all_probs, all_preds, all_labels = [], [], []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(device)
            prob = torch.sigmoid(model(imgs)[:, 0]).cpu().numpy()
            pred = (prob > 0.5).astype(np.int64)
            all_probs.append(prob.flatten())
            all_preds.append(pred.flatten())
            all_labels.append(masks.numpy().flatten())

    all_probs  = np.concatenate(all_probs)
    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    TP = ((all_preds == 1) & (all_labels == 1)).sum()
    FP = ((all_preds == 1) & (all_labels == 0)).sum()
    TN = ((all_preds == 0) & (all_labels == 0)).sum()
    FN = ((all_preds == 0) & (all_labels == 1)).sum()

    dice = 2 * TP / (2 * TP + FP + FN + 1e-8)
    sens = TP / (TP + FN + 1e-8)
    spec = TN / (TN + FP + 1e-8)
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except Exception:
        auc = 0.0

    return {'Dice': float(dice), 'Sensitivity': float(sens),
            'Specificity': float(spec), 'AUC': float(auc)}


test_model = build_model()
n_params   = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
print(f'U-Net (ResNet34) 파라미터 수: {n_params:,}')
del test_model
print('모델 + 유틸리티 함수 준비 완료')

In [ ]:
# === Cell 4: Optuna alpha/gamma 탐색 (학습 함수 포함) ===

def train_model(
    loss_name,
    alpha=1.0,
    gamma=2.0,
    epochs=50,
    lr=1e-4,
    subset_ratio=1.0,
    tag='',
):
    """
    U-Net(ResNet34) 학습 함수.
    WMH: 극심한 불균형 (BG:FG > 100:1) → LWCE 계열 효과 극대화 환경
    """
    model     = build_model()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = get_loss_function(loss_name, class_counts=class_counts, alpha=alpha, gamma=gamma)

    name = f'{loss_name}_alpha{alpha:.2f}' if alpha != 1.0 else loss_name
    if tag:
        name = f'{tag}_{name}'

    print(f"\n{'='*60}\nU-Net(ResNet34) + {name}  (epochs={epochs})\n{'='*60}")

    if subset_ratio < 1.0:
        n = max(1, int(len(train_loader.dataset) * subset_ratio))
        sub_ds = torch.utils.data.Subset(
            train_loader.dataset,
            random.sample(range(len(train_loader.dataset)), n)
        )
        loader = DataLoader(sub_ds, batch_size=BATCH_SIZE,
                            shuffle=True, num_workers=NUM_WORKERS)
    else:
        loader = train_loader

    history    = {'loss': [], 'val_dice': []}
    best_dice  = 0.0
    save_path  = f'/tmp/best_unet_wmh_{name}.pth'

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for imgs, masks in tqdm(loader, desc=f'Ep{epoch+1:02d}/{epochs}', leave=False):
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            logits_2ch = to_2ch_logits(model(imgs))
            loss = criterion(logits_2ch, masks)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()
        avg_loss = epoch_loss / len(loader)
        val_dice = compute_val_dice(model, val_loader)

        history['loss'].append(avg_loss)
        history['val_dice'].append(val_dice)

        print(f'Ep{epoch+1:02d} | Loss: {avg_loss:.4f} | Val Dice: {val_dice:.4f}', end='')
        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.state_dict(), save_path)
            print('  <- Best!', end='')
        print()

    model.load_state_dict(torch.load(save_path, weights_only=True))
    print(f'최고 Val Dice: {best_dice:.4f}')
    return model, history, best_dice


print('train_model() 함수 준비 완료')

In [ ]:
# === Cell 5: Optuna alpha/gamma 탐색 ===
#
# WMH 극심한 불균형 특성상 큰 alpha가 필요할 수 있음:
#   plwce: alpha 범위 2.5 ~ 20.0 (LiTS보다 넓게)
#   pwce:  alpha 범위 0.2 ~ 3.0
import traceback

os.environ['TQDM_DISABLE'] = '1'

ALPHA_LOW_PLWCE,  ALPHA_HIGH_PLWCE  = 2.5, 20.0
ALPHA_LOW_PWCE,   ALPHA_HIGH_PWCE   = 0.2,  3.0
PROXY_EPOCHS = 8
PROXY_SUBSET = 0.15
N_TRIALS     = 20
N_TRIALS_PF  = 40  # PLWCE+Focal: 파라미터 2개(alpha,gamma)이므로 2배


def make_objective(loss_name, alpha_low, alpha_high):
    def objective(trial):
        alpha = trial.suggest_float('alpha', alpha_low, alpha_high)
        try:
            _, _, dice = train_model(
                loss_name    = loss_name,
                alpha        = alpha,
                epochs       = PROXY_EPOCHS,
                subset_ratio = PROXY_SUBSET,
                tag          = f'trial{trial.number}',
            )
            return dice
        except Exception as e:
            print(f'Trial {trial.number} 실패: {e}')
            traceback.print_exc()
            return 0.0
    return objective


print(f'[Optuna] PLWCE alpha 탐색  (범위: {ALPHA_LOW_PLWCE}~{ALPHA_HIGH_PLWCE}, {N_TRIALS} trials)')
sampler_plwce = optuna.samplers.GridSampler({'alpha': np.linspace(ALPHA_LOW_PLWCE, ALPHA_HIGH_PLWCE, N_TRIALS).tolist()})
study_plwce = optuna.create_study(
    direction  = 'maximize',
    study_name = 'unet_wmh_plwce_alpha',
    sampler    = sampler_plwce,
)
study_plwce.optimize(make_objective('plwce_dice', ALPHA_LOW_PLWCE, ALPHA_HIGH_PLWCE), n_trials=N_TRIALS)
best_alpha_plwce = study_plwce.best_params['alpha']
print(f'[PLWCE] 최적 alpha = {best_alpha_plwce:.4f}  (Val Dice = {study_plwce.best_value:.4f})')

print(f'\n[Optuna] PWCE alpha 탐색  (범위: {ALPHA_LOW_PWCE}~{ALPHA_HIGH_PWCE}, {N_TRIALS} trials)')
sampler_pwce = optuna.samplers.GridSampler({'alpha': np.linspace(ALPHA_LOW_PWCE, ALPHA_HIGH_PWCE, N_TRIALS).tolist()})
study_pwce = optuna.create_study(
    direction  = 'maximize',
    study_name = 'unet_wmh_pwce_alpha',
    sampler    = sampler_pwce,
)
study_pwce.optimize(make_objective('pwce_dice', ALPHA_LOW_PWCE, ALPHA_HIGH_PWCE), n_trials=N_TRIALS)
best_alpha_pwce = study_pwce.best_params['alpha']
print(f'[PWCE]  최적 alpha = {best_alpha_pwce:.4f}  (Val Dice = {study_pwce.best_value:.4f})')

# --- PLWCE+Focal alpha + gamma 공동 탐색 ---
ALPHA_LOW_PF, ALPHA_HIGH_PF = 2.5, 20.0
GAMMA_LOW_PF, GAMMA_HIGH_PF = 0.5,  5.0

print(f'\n[Optuna] PLWCE+Focal alpha+gamma 탐색  '
      f'(alpha: {ALPHA_LOW_PF}~{ALPHA_HIGH_PF}, gamma: {GAMMA_LOW_PF}~{GAMMA_HIGH_PF}, {N_TRIALS_PF} trials)')

def objective_pf(trial):
    alpha = trial.suggest_float('alpha', ALPHA_LOW_PF, ALPHA_HIGH_PF)
    gamma = trial.suggest_float('gamma', GAMMA_LOW_PF, GAMMA_HIGH_PF)
    try:
        _, _, dice = train_model(
            loss_name    = 'plwce_focal_dice',
            alpha        = alpha,
            gamma        = gamma,
            epochs       = PROXY_EPOCHS,
            subset_ratio = PROXY_SUBSET,
            tag          = f'trial{trial.number}',
        )
        return dice
    except Exception as e:
        print(f'Trial {trial.number} 실패: {e}')
        traceback.print_exc()
        return 0.0

N_ALPHA_GRID = 8  # 8x5=40 grid
N_GAMMA_GRID = 5
sampler_pf = optuna.samplers.GridSampler({
    'alpha': np.linspace(ALPHA_LOW_PF, ALPHA_HIGH_PF, N_ALPHA_GRID).tolist(),
    'gamma': np.linspace(GAMMA_LOW_PF, GAMMA_HIGH_PF, N_GAMMA_GRID).tolist(),
})
study_pf = optuna.create_study(
    direction  = 'maximize',
    study_name = f'unet_{DOMAIN}_plwce_focal_alpha_gamma',
    sampler    = sampler_pf,
)
study_pf.optimize(objective_pf, n_trials=N_TRIALS_PF)
best_alpha_pf = study_pf.best_params['alpha']
best_gamma_pf = study_pf.best_params['gamma']
print(f'[PLWCE+Focal] 최적 alpha={best_alpha_pf:.4f}, gamma={best_gamma_pf:.4f}  '
      f'(Val Dice = {study_pf.best_value:.4f})')

optuna_results = {
    'plwce': {
        'best_alpha': best_alpha_plwce,
        'best_proxy_dice': study_plwce.best_value,
        'trials': [{'number': t.number, 'alpha': t.params.get('alpha'), 'value': t.value}
                   for t in study_plwce.trials if t.value is not None],
    },
    'pwce': {
        'best_alpha': best_alpha_pwce,
        'best_proxy_dice': study_pwce.best_value,
        'trials': [{'number': t.number, 'alpha': t.params.get('alpha'), 'value': t.value}
                   for t in study_pwce.trials if t.value is not None],
    },
    'plwce_focal': {
        'best_alpha': best_alpha_pf,
        'best_gamma': best_gamma_pf,
        'best_proxy_dice': study_pf.best_value,
        'trials': [{'number': t.number, 'alpha': t.params.get('alpha'),
                    'gamma': t.params.get('gamma'), 'value': t.value}
                   for t in study_pf.trials if t.value is not None],
    },
}
with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_optuna_results.json'), 'w') as f:
    json.dump(optuna_results, f, indent=2, ensure_ascii=False)
print(f'Optuna 결과 저장: {RESULTS_DIR}/{DOMAIN}_optuna_results.json')

# --- PLWCE / PWCE alpha 탐색 결과 시각화 ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, study, sname, a_range in [
    (axes[0], study_plwce, 'PLWCE', f'{ALPHA_LOW_PLWCE}~{ALPHA_HIGH_PLWCE}'),
    (axes[1], study_pwce,  'PWCE',  f'{ALPHA_LOW_PWCE}~{ALPHA_HIGH_PWCE}'),
]:
    trials = [t for t in study.trials if t.value is not None]
    alphas = [t.params['alpha'] for t in trials]
    values = [t.value for t in trials]
    best_a = study.best_params['alpha']
    best_v = study.best_value
    ax.scatter(alphas, values, alpha=0.5, s=40, label='Trials')
    ax.axvline(best_a, color='red', linestyle='--', label=f'Best alpha={best_a:.2f}')
    ax.scatter([best_a], [best_v], color='red', s=100, zorder=5)
    ax.set_xlabel('alpha'); ax.set_ylabel('Val Dice (proxy)')
    ax.set_title(f'{sname} alpha 탐색 (WMH, 범위 {a_range})')
    ax.legend(); ax.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{DOMAIN}_optuna_search.png'), dpi=100)
plt.show()

# --- PLWCE+Focal: alpha vs gamma 2D 탐색 결과 시각화 ---
fig_pf, ax_pf = plt.subplots(1, 1, figsize=(7, 5))
pf_trials = [t for t in study_pf.trials if t.value is not None]
pf_alphas = [t.params['alpha'] for t in pf_trials]
pf_gammas = [t.params['gamma'] for t in pf_trials]
pf_values = [t.value for t in pf_trials]
sc = ax_pf.scatter(pf_alphas, pf_gammas, c=pf_values, cmap='viridis', alpha=0.7, s=60)
ax_pf.scatter([best_alpha_pf], [best_gamma_pf], color='red', s=150, zorder=5,
              marker='*', label=f'Best α={best_alpha_pf:.2f}, γ={best_gamma_pf:.2f}')
plt.colorbar(sc, ax=ax_pf, label='Val Dice')
ax_pf.set_xlabel('alpha'); ax_pf.set_ylabel('gamma')
ax_pf.set_title('PLWCE+Focal alpha+gamma 탐색')
ax_pf.legend(fontsize=8); ax_pf.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{DOMAIN}_optuna_search_pf.png'), dpi=100)
plt.show()
print(f'PLWCE+Focal 탐색 결과 저장: {RESULTS_DIR}/{DOMAIN}_optuna_search_pf.png')

In [ ]:
# === Cell 6: 전체 Loss 비교 학습 ===

FINAL_EPOCHS = 50
FINAL_LR     = 1e-4

# --- Optuna 결과 로드 (미실행 시 기본값) ---
try:
    _ = best_alpha_plwce
except NameError:
    try:
        with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_optuna_results.json')) as f:
            d = json.load(f)
        best_alpha_plwce = d['plwce']['best_alpha']
        best_alpha_pwce  = d['pwce']['best_alpha']
        best_alpha_pf    = d.get('plwce_focal', {}).get('best_alpha', 7.0)
        best_gamma_pf    = d.get('plwce_focal', {}).get('best_gamma', 2.0)
        print(f'Optuna 결과 로드: PLWCE alpha={best_alpha_plwce:.4f}')
    except FileNotFoundError:
        best_alpha_plwce = 10.0
        best_alpha_pwce  = 0.8
        print('Optuna 미실행 → 기본값 사용 (WMH 극심한 불균형: PLWCE alpha=10.0)')

experiments = [
    ('ce_dice',          1.0,              2.0,             'CE+Dice              (기준선)'),
    ('wce_dice',         1.0,              2.0,             'WCE+Dice'),
    ('lwce_dice',        1.0,              2.0,             'LWCE+Dice'),
    ('plwce_dice',       best_alpha_plwce, 2.0,             f'PLWCE+Dice           (alpha={best_alpha_plwce:.2f})'),
    ('cb_dice',          1.0,              2.0,             'CB+Dice'),
    ('plwce_focal_dice', best_alpha_pf,    best_gamma_pf,   f'PLWCE+Focal+Dice     (α={best_alpha_pf:.2f}, γ={best_gamma_pf:.2f})'),
]

all_results = {}
for loss_name, alpha, gamma, label in experiments:
    model, history, best_dice = train_model(
        loss_name = loss_name,
        alpha     = alpha,
        gamma     = gamma,
        epochs    = FINAL_EPOCHS,
        lr        = FINAL_LR,
        tag       = 'final',
    )
    all_results[label] = {
        'model':     model,
        'history':   history,
        'best_dice': best_dice,
        'loss_name': loss_name,
        'alpha':     alpha,
        'gamma':     gamma,
    }

print('\n' + '='*50)
print('[WMH Loss 비교 실험 요약 — Val Dice]')
print(f"{'Loss':<35} {'Best Val Dice':>13}")
print('-' * 50)
for label, v in all_results.items():
    print(f"{label:<35} {v['best_dice']:>13.4f}")

In [ ]:
# === Cell 6: 평가 및 결과 저장 (시각화) ===

COLORS = ['#4878D0', '#EE854A', '#6ACC65', '#D65F5F', '#B47CC7']

# --- 학습 곡선 ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for i, (label, v) in enumerate(all_results.items()):
    h = v['history']
    ax1.plot(h['loss'],     label=label, color=COLORS[i % len(COLORS)])
    ax2.plot(h['val_dice'], label=label, color=COLORS[i % len(COLORS)])

ax1.set_title('Train Loss'); ax1.set_xlabel('Epoch')
ax1.legend(fontsize=7); ax1.grid(True)
ax2.set_title('Val Dice (WMH Lesion)'); ax2.set_xlabel('Epoch')
ax2.legend(fontsize=7); ax2.grid(True)

plt.suptitle('WMH — U-Net(ResNet34) 학습 곡선 비교', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{DOMAIN}_training_curves.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'학습 곡선 저장: {RESULTS_DIR}/{DOMAIN}_training_curves.png')

# --- 예측 결과 시각화 (4열: FLAIR / GT / Prob Map / Pred) ---
best_label = max(all_results, key=lambda k: all_results[k]['best_dice'])
best_model = all_results[best_label]['model']
best_model.eval()
print(f'\n시각화 모델: {best_label}  (Val Dice={all_results[best_label]["best_dice"]:.4f})')

val_ds      = WMHDataset(val_files, augment=False)
# WMH 있는 슬라이스만 선택해서 시각화
wmh_val_idx = [i for i in range(len(val_ds)) if val_ds[i][1].sum() > 0]
vis_indices = random.sample(wmh_val_idx, min(4, len(wmh_val_idx)))

fig, axes = plt.subplots(len(vis_indices), 4, figsize=(18, len(vis_indices) * 4))
if len(vis_indices) == 1:
    axes = axes[np.newaxis, :]

for row, idx in enumerate(vis_indices):
    img_t, mask_t = val_ds[idx]
    flair_ch = img_t[0].numpy()  # FLAIR 채널

    with torch.no_grad():
        logit = best_model(img_t.unsqueeze(0).to(device))
        prob  = torch.sigmoid(logit[0, 0]).cpu().numpy()
        pred  = (prob > 0.5).astype(np.uint8)

    axes[row, 0].imshow(flair_ch, cmap='gray')
    axes[row, 0].set_title('FLAIR MRI'); axes[row, 0].axis('off')
    axes[row, 1].imshow(mask_t.numpy(), cmap='hot', vmin=0, vmax=1)
    axes[row, 1].set_title('Ground Truth (WMH=White)'); axes[row, 1].axis('off')
    axes[row, 2].imshow(prob, cmap='jet', vmin=0, vmax=1)
    axes[row, 2].set_title('WMH Probability Map'); axes[row, 2].axis('off')
    axes[row, 3].imshow(pred, cmap='hot', vmin=0, vmax=1)
    axes[row, 3].set_title(f'Prediction ({best_label.split("(")[0].strip()[:12]})')
    axes[row, 3].axis('off')

plt.suptitle(f'WMH — 예측 결과 시각화 ({best_label})', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{DOMAIN}_prediction_vis.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'예측 결과 저장: {RESULTS_DIR}/{DOMAIN}_prediction_vis.png')

In [ ]:
# === Cell 6: 평가 및 결과 저장 (정량 평가 + JSON + Excel) ===

print('\n[전체 모델 종합 평가 — Test Set]')
print(f"{'Loss':<35} {'Dice':>7} {'Sens':>7} {'Spec':>7} {'AUC':>7}")
print('-' * 65)

final_results = {}
for label, v in all_results.items():
    metrics = compute_val_metrics(v['model'], test_loader)
    final_results[label] = {
        'loss_name':     v['loss_name'],
        'alpha':         v['alpha'],
        'best_val_dice': v['best_dice'],
        **metrics,
    }
    print(
        f"{label:<35} "
        f"{metrics['Dice']:>7.4f} "
        f"{metrics['Sensitivity']:>7.4f} "
        f"{metrics['Specificity']:>7.4f} "
        f"{metrics['AUC']:>7.4f}"
    )

# --- 바차트 비교 ---
metric_keys = ['Dice', 'Sensitivity', 'Specificity', 'AUC']
labels_     = list(final_results.keys())

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, mkey in zip(axes, metric_keys):
    scores = [final_results[lb][mkey] for lb in labels_]
    bars   = ax.bar(range(len(labels_)), scores, color=COLORS[:len(labels_)], alpha=0.85)
    ax.set_xticks(range(len(labels_)))
    ax.set_xticklabels(
        [lb.split('(')[0].strip()[:12] for lb in labels_],
        rotation=30, ha='right', fontsize=8
    )
    ax.set_title(mkey); ax.set_ylim(0, 1.05); ax.grid(axis='y', alpha=0.4)
    best_idx = int(np.argmax(scores))
    bars[best_idx].set_edgecolor('red'); bars[best_idx].set_linewidth(2.5)
    for bar, score in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{score:.3f}', ha='center', va='bottom', fontsize=7)

plt.suptitle('WMH — Loss별 최종 평가 지표 비교 (Test Set)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{DOMAIN}_final_metrics.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'평가 차트 저장: {RESULTS_DIR}/{DOMAIN}_final_metrics.png')

# --- JSON 저장 ---
save_data = {
    'domain':       'WMH 2017 Brain White Matter Hyperintensity Segmentation',
    'model':        'U-Net (ResNet34, ImageNet pretrained)',
    'sota_ref':     {
        'nnUNet_3D': {'Dice': 0.800, 'Sensitivity': 0.79, 'Specificity': 0.99},
        '2D_UNet':   {'Dice': '0.75~0.79'},
    },
    'num_classes':  NUM_CLASSES,
    'class_counts': {n: int(c) for n, c in zip(CLASS_NAMES, class_counts)},
    'imbalance':    {'BG_WMH': round(class_counts[0] / class_counts[1], 1)},
    'input_channels':   '3 (FLAIR, T1, FLAIR)',
    'bg_only_ratio':    BG_ONLY_RATIO,
    'train_slices': len(tr_files),
    'val_slices':   len(val_files),
    'final_epochs': FINAL_EPOCHS,
    'results': {
        k: {mk: float(mv) if isinstance(mv, (float, np.floating)) else mv
            for mk, mv in v.items() if mk != 'model'}
        for k, v in final_results.items()
    },
    'best_model': max(final_results, key=lambda k: final_results[k]['Dice']),
}
with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_final_results.json'), 'w', encoding='utf-8') as f:
    json.dump(save_data, f, indent=2, ensure_ascii=False)
print(f'JSON 저장: {RESULTS_DIR}/{DOMAIN}_final_results.json')

# --- Excel 저장 ---
summary_rows = []
for label, v in final_results.items():
    summary_rows.append({
        'Loss_Function':   label,
        'loss_name':       v['loss_name'],
        'alpha':           round(float(v['alpha']), 4),
        'Best_Val_Dice':   round(v['best_val_dice'], 4),
        'Test_Dice':        round(v['Dice'],        4),
        'Test_Sensitivity': round(v['Sensitivity'], 4),
        'Test_Specificity': round(v['Specificity'], 4),
        'Test_AUC':         round(v['AUC'],         4),
        'BG_WMH_ratio':    round(class_counts[0] / class_counts[1], 1),
        'BG_only_ratio':   BG_ONLY_RATIO,
        'epochs':          FINAL_EPOCHS,
        'model':           'U-Net (ResNet34)',
    })
df_summary = pd.DataFrame(summary_rows)

history_rows = []
for label, v in all_results.items():
    for ep, (loss, dice) in enumerate(
        zip(v['history']['loss'], v['history']['val_dice']), 1
    ):
        history_rows.append({
            'Loss_Function': label,
            'Epoch':         ep,
            'Train_Loss':    round(loss, 6),
            'Val_Dice':      round(dice, 6),
        })
df_history = pd.DataFrame(history_rows)

excel_path = os.path.join(RESULTS_DIR, f'{DOMAIN}_final_results.xlsx')
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_summary.to_excel(writer, sheet_name='Summary',          index=False)
    df_history.to_excel(writer, sheet_name='Training_History', index=False)
print(f'Excel 저장: {excel_path}')

print(f"\n최고 모델: {save_data['best_model']}")
print(f'BG : WMH = {save_data["imbalance"]["BG_WMH"]:>8.1f} : 1  (극심한 불균형)')